# J5-01 · Middlewares LangChain 1.x : des contrôles autour de la boucle de l'agent

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 5 · créneau 08h30–10h00 · ⏱️ 1h30 (dont 45 min d'exercice)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Situer les **points d'accroche** d'un middleware dans la boucle d'un `create_agent` · 2. Utiliser les middlewares fournis (PII, plafonds d'appels, retries, repli de modèle) et en connaître les **limites** · 3. Écrire son propre middleware `wrap_tool_call`, **synchrone et asynchrone** · 4. Préparer l'exercice 5.1 (liste blanche + journal d'audit) |
| 🧱 **Fil rouge** | Le spécialiste Diagnostic de HelpDesk-Ops reçoit ses premiers contrôles de production |
| ✅ **Prérequis** | J1 à J4 ; versions vérifiées : `langchain` 1.4.2 (lock figé) |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Où s'accroche un middleware ?

### [Théorie] Les contrôles de l'aéroport

Un passager (la demande) traverse des contrôles **avant** l'embarquement, ses bagages passent **au scanner** pendant le voyage, et la douane le contrôle **à l'arrivée**. Aucun contrôle ne pilote l'avion : ils encadrent le trajet. Un middleware fait pareil autour de la boucle « modèle ⇄ outils » d'un `create_agent` : il **observe**, **modifie** ou **interrompt**, sans réécrire l'agent.

```mermaid
flowchart LR
    A([demande]) --> BA[before_agent]
    BA --> BM[before_model]
    BM --> WM[[wrap_model_call<br/>le modèle]]
    WM --> AM[after_model]
    AM -->|appel d'outil| WT[[wrap_tool_call<br/>l'outil]]
    WT --> BM
    AM -->|réponse finale| AA[after_agent]
    AA --> R([réponse])
```

| Point d'accroche | Appelé | Usage typique |
|---|---|---|
| `before_agent` / `after_agent` | une fois par exécution | charger un contexte, auditer la réponse finale |
| `before_model` / `after_model` | à chaque tour de boucle | retirer des données personnelles, compter, arrêter |
| `wrap_model_call` | autour de l'appel au modèle | retry, repli vers un autre modèle, choix dynamique des outils |
| `wrap_tool_call` | autour de chaque appel d'outil | liste blanche, retry, journal d'audit, validation humaine |
| `dynamic_prompt` | à la construction du prompt système | prompt selon l'utilisateur, la date, le contexte |

Chaque point existe en **décorateur** (`@before_model`, `@wrap_tool_call`…) pour un contrôle simple, ou en méthode d'une sous-classe d'`AgentMiddleware` pour un contrôle qui a des réglages ou un état.

## 3. L'ordre d'exécution, observé

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import after_model, before_model, wrap_tool_call
from langchain_core.messages import HumanMessage

from helpdesk_ops.outils import statut_serveur

ordre = []


def temoin(nom: str) -> list:
    """Deux middlewares témoins qui notent leur passage."""
    @before_model(name=f"avant_{nom}")
    def avant(state, runtime):
        ordre.append(f"{nom}.before_model")

    @after_model(name=f"apres_{nom}")
    def apres(state, runtime):
        ordre.append(f"{nom}.after_model")

    @wrap_tool_call(name=f"outil_{nom}")
    def autour(request, handler):
        ordre.append(f"{nom}.wrap_tool_call ↓")
        resultat = handler(request)
        ordre.append(f"{nom}.wrap_tool_call ↑")
        return resultat
    return [avant, apres, autour]


agent = create_agent(llm, [statut_serveur], system_prompt="Tu es l'agent support N1 de NordTech. Réponds en une phrase.",
                     middleware=[*temoin("A"), *temoin("B")])
with propagate_attributes(session_id="j5-01-middleware", tags=["jour-5", "middleware"]):
    etat = agent.invoke({"messages": [HumanMessage("srv-erp-02 est-il en ligne ?")]}, config={"callbacks": [langfuse_handler]})
print(etat["messages"][-1].content, "\n")
print("\n".join(ordre))

**👀 Ce que vous devez observer :** les `before_model` dans l'ordre de la liste (A puis B), les `after_model` dans l'ordre **inverse** (B puis A), et les `wrap_tool_call` **emboîtés** : A est l'enveloppe extérieure (A ↓, B ↓, outil, B ↑, A ↑). Retenez la règle : **le premier de la liste est le plus à l'extérieur**. Un middleware de journal d'audit se place donc en tête, pour voir aussi ce que les suivants refusent.

## 4. Les middlewares fournis (vérifiés dans `langchain` 1.4.2)

| Middleware | Ce qu'il fait | Dans HelpDesk-Ops |
|---|---|---|
| `PIIMiddleware` | détecte e-mails, cartes, IP, MAC, URL, ou **vos** motifs ; `redact`, `mask`, `hash` ou `block` | données personnelles retirées avant le modèle |
| `ModelCallLimitMiddleware` / `ToolCallLimitMiddleware` | plafonds par exécution (`run_limit`) ou par conversation (`thread_limit`) | un agent qui boucle s'arrête (red-team ASI08) |
| `ModelRetryMiddleware` / `ToolRetryMiddleware` | nouvelles tentatives avec attente exponentielle | supervision instable (J2) |
| `ToolErrorMiddleware` | transforme les exceptions choisies en message d'erreur pour le modèle | panne réseau lisible par l'agent |
| `ModelFallbackMiddleware` | modèle de repli si le principal échoue | Ollama surchargé → modèle plus petit |
| `HumanInTheLoopMiddleware` | validation humaine avant certains outils (vu en J2) | ticket P1 |
| `SummarizationMiddleware` / `ContextEditingMiddleware` | résumer l'historique / retirer d'anciens résultats d'outils | longues conversations |
| `LLMToolSelectorMiddleware` | un LLM choisit les outils utiles avant l'appel principal | beaucoup d'outils MCP |
| `TodoListMiddleware`, `ShellToolMiddleware`, `FilesystemFileSearchMiddleware` | plan, shell, recherche de fichiers (briques de Deep Agents, J3) | **non** : pas de shell pour un agent N1 |

> 💡 **Le « Tips » du Formateur :** la liste évolue vite (`ToolErrorMiddleware` et `TracePolicy` sont récents). Vérifiez dans **votre** version : `import langchain.agents.middleware as m; dir(m)`.

## 5. `PIIMiddleware` : il protège le modèle, pas la trace

Le `PIIMiddleware` agit en `before_model` : il réécrit le dernier message de l'utilisateur (et, si demandé, les résultats d'outils) **avant** le modèle. Mais la demande a déjà été tracée à l'entrée du graphe. On le vérifie avec une adresse IP, que le masque Langfuse de J1 ne connaît pas.

In [ ]:
from langchain.agents.middleware import PIIMiddleware

from helpdesk_ops.securite import middlewares_pii

agent_pii = create_agent(llm, [statut_serveur], system_prompt="Tu es l'agent support N1 de NordTech. Réponds en une phrase.",
                         middleware=[*middlewares_pii(), PIIMiddleware("ip", strategy="redact", apply_to_input=True)])
DEMANDE = "Je suis julie.martin@nordtech.be, mon poste a l'adresse 10.12.4.27 : srv-erp-02 est-il en ligne ?"

with propagate_attributes(session_id="j5-01-pii", tags=["jour-5", "pii"]):
    with langfuse.start_as_current_observation(name="demo-pii", as_type="agent", input=DEMANDE) as racine:
        etat = agent_pii.invoke({"messages": [HumanMessage(DEMANDE)]}, config={"callbacks": [langfuse_handler]})
        trace_pii = langfuse.get_current_trace_id()
print("dans l'état (et donc le checkpoint) :", etat["messages"][0].content)
print("réponse :", etat["messages"][-1].content)
langfuse.flush()

In [ ]:
import time

time.sleep(5)   # le temps que Langfuse ingère la trace
observations = langfuse.api.observations.get_many(trace_id=trace_pii, fields="core,basic,io", limit=100).data
for o in sorted(observations, key=lambda o: o.start_time):
    texte = f"{o.input}"
    print(f"{o.type:<11} {o.name[:28]:<28} e-mail en clair : {'julie.martin' in texte!s:<5}  IP en clair : {'10.12.4.27' in texte}")

**👀 Ce que vous devez observer :** dans l'état, l'e-mail et l'IP sont remplacés par `[REDACTED_…]` : le modèle ne les a jamais vus, et le checkpoint non plus. Dans la trace, l'e-mail est masqué partout (c'est le `mask=` de J1, appliqué à l'envoi), mais **l'IP apparaît en clair** dans six observations (mesuré en préparation) : la racine, le graphe, et même l'**entrée des middlewares PII** eux-mêmes, puisqu'on trace ce qu'ils reçoivent. À partir de l'appel au modèle, elle a disparu. Le `PIIMiddleware` protège tout ce qui vient **après** lui, rien de ce qui a été capturé avant.

La règle : **trois protections, trois endroits**. `PIIMiddleware` pour le modèle et l'état ; `mask=` du client pour les traces ; et la minimisation dans l'interface (ne pas demander l'information). Un motif ajouté à l'un doit l'être aux deux autres.

## 6. Plafonds, retries, repli : la robustesse en une ligne chacun

In [ ]:
from langchain.agents.middleware import ModelCallLimitMiddleware, ModelFallbackMiddleware
from langchain_ollama import ChatOllama

# Un modèle « en panne » (nom inexistant : Ollama répond 404), avec un repli vers le petit modèle
modele_en_panne = ChatOllama(model="modele-inexistant:latest", base_url=os.environ["OLLAMA_BASE_URL"])
agent_repli = create_agent(modele_en_panne, [statut_serveur], system_prompt="Tu es l'agent support N1. Réponds en une phrase.",
                           middleware=[ModelFallbackMiddleware(creer_llm("small")),
                                       ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")])
debut = time.perf_counter()
with propagate_attributes(session_id="j5-01-repli", tags=["jour-5", "repli"]):
    etat = agent_repli.invoke({"messages": [HumanMessage("srv-erp-02 est-il en ligne ?")]}, config={"callbacks": [langfuse_handler]})
print(f"{time.perf_counter() - debut:.0f} s →", etat["messages"][-1].content)
print("modèle qui a répondu :", etat["messages"][-1].response_metadata.get("model_name") or etat["messages"][-1].response_metadata.get("model"))

**👀 Ce que vous devez observer :** la réponse arrive malgré le modèle principal en erreur : le repli `granite4.1:3b` a pris le relais. Dans Langfuse, la génération en **ERROR** (404 du modèle inexistant) est suivie de celle du modèle de repli : la panne est **invisible pour l'utilisateur, visible pour l'équipe**. C'est le bon partage.

Un piège constaté en préparant le cours : quand `ModelCallLimitMiddleware(exit_behavior="end")` coupe l'agent, le message final est **en anglais** (« Model call limits exceeded: run limit (4/4) ») et part tel quel à l'utilisateur. Le jalon J5 le remplace par un message de repli en français, dans son nœud de contrôle de sortie.

## 7. Écrire son middleware : attention au synchrone et à l'asynchrone

Le décorateur `@wrap_tool_call` sur une fonction **synchrone** fonctionne avec `invoke`… et échoue avec `ainvoke`. Le coordinateur de J3 délègue en asynchrone : c'est exactement ce qui s'est produit en préparant le jalon J5.

In [ ]:
from helpdesk_ops.reseau import ping_serveur


@wrap_tool_call(name="journal_simple")
def journal_simple(request, handler):
    print("   appel :", request.tool_call["name"], request.tool_call["args"])
    return handler(request)


agent_journal = create_agent(llm, [ping_serveur], system_prompt="Tu es l'agent support N1. Réponds en une phrase.",
                             middleware=[journal_simple])
try:
    await agent_journal.ainvoke({"messages": [HumanMessage("srv-erp-02 répond-il au ping ?")]})   # Jupyter : await direct
except NotImplementedError as e:
    print("❌", type(e).__name__, ":", str(e)[:160], "…")

**👀 Ce que vous devez observer :** `NotImplementedError : Asynchronous implementation of awrap_tool_call is not available…`. Deux corrections : décorer une fonction `async def` (alors seul l'asynchrone marche), ou, pour un middleware réutilisable, une **sous-classe** d'`AgentMiddleware` qui implémente `wrap_tool_call` **et** `awrap_tool_call`. C'est la forme demandée dans l'exercice 5.1.

> 💡 **Le « Tips » du Formateur :** dans Jupyter, une boucle asyncio tourne déjà : on écrit `await agent.ainvoke(...)` directement dans la cellule. `asyncio.run(...)` y échoue (« cannot be called from a running event loop ») ; dans un script, c'est l'inverse.

## 8. 🧪 Exercice 5.1 (45 min) : liste blanche et journal d'audit

`exercices/enonces/Ex5_1_liste_blanche_audit.ipynb` : un middleware qui n'autorise que les outils listés, renvoie un refus **lisible par le modèle** pour les autres, et écrit chaque décision dans un journal JSON Lines (arguments masqués). Branché sur un spécialiste qui reçoit **tous** les outils MCP du J3, y compris `reinitialiser_poste`.

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Placer un contrôle dans la boucle | `before_model`, `after_model`, `wrap_model_call`, `wrap_tool_call`, `before_agent`, `after_agent` |
| L'ordre d'exécution | `before_*` dans l'ordre de la liste, `after_*` à l'envers, `wrap_*` emboîtés (le 1ᵉʳ à l'extérieur) |
| Retirer des données personnelles avant le modèle | `PIIMiddleware(type, strategy="redact", detector=…)`, **et** `mask=` pour les traces |
| Borner, réessayer, se replier | `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware`, `ToolRetryMiddleware`, `ModelFallbackMiddleware` |
| Écrire un middleware réutilisable | sous-classe d'`AgentMiddleware` avec `wrap_tool_call` **et** `awrap_tool_call` |

### ✅ Quiz express

1. Deux middlewares A et B dans cet ordre : lequel voit le résultat de l'outil en dernier ?
2. Le `PIIMiddleware` suffit-il à garder les e-mails hors de Langfuse ?
3. Pourquoi un `@wrap_tool_call` sur une fonction synchrone casse-t-il le coordinateur de J3 ?
4. Un modèle de repli plus petit répond : qu'est-ce qui doit rester visible, et pour qui ?

<details>
<summary>Réponses</summary>

1. A : il est l'enveloppe extérieure ; B rend le résultat à A.
2. Non : il agit avant le modèle, après la capture de l'entrée par la trace. Il faut aussi `mask=` côté Langfuse (et ne pas demander l'information).
3. Le coordinateur appelle ses outils en asynchrone ; le middleware n'a pas d'`awrap_tool_call` : `NotImplementedError`.
4. La panne et le repli doivent être visibles pour l'équipe (génération en ERROR, modèle de repli dans la trace) ; l'utilisateur reçoit une réponse normale.
</details>